# 모델 로드

In [2]:
import os
os.environ["TORCH_COMPILE_DISABLE"] = "1"

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 모델 및 토크나이저 로드 (3개 클래스로 설정 반영)
model_name = "beomi/KcELECTRA-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

config.json:   0%|          | 0.00/514 [00:00<?, ?B/s]

C:\Project\Python_Source\AI01\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Win11Pro\.cache\huggingface\hub\models--beomi--KcELECTRA-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.78M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base
Key                                               | Status     | Details
--------------------------------------------------+------------+--------
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |        
discriminator_predictions.dense.bias              | UNEXPECTED |        
discriminator_predictions.dense_prediction.weight | UNEXPECTED |        
discriminator_predictions.dense.weight            | UNEXPECTED |        
classifier.out_proj.bias                          | MISSING    |        
classifier.dense.weight                           | MISSING    |        
classifier.out_proj.weight                        | MISSING    |        
classifier.dense.bias                             | MISSING    |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing

# 데이터 로드

In [20]:
import pandas as pd
import numpy as np

file_path = r"C:\Project\Python_Source\AI01\mini_project_09\018.감성대화\Training_221115_add\원천데이터\감성대화말뭉치(최종데이터)_Training.csv"
df = pd.read_csv(file_path)

In [21]:
df.groupby("감정_대분류").describe()

Unnamed: 0                                                       \
            count          mean           std   min       25%      50%   
감정_대분류                                                                   
기쁨         6126.0  18824.161770  10984.588174   8.0   9343.25  18492.5   
당황         8756.0  27619.208429  15076.728776  15.0  14736.75  28368.5   
분노         9160.0  26962.584389  15038.799244   1.0  13944.75  27737.5   
불안         9320.0  25503.250000  15266.941101   9.0  11916.50  25056.5   
상처         9143.0  27147.280324  15095.007287  26.0  14018.50  27740.0   
슬픔         9125.0  26611.344000  14990.782401  19.0  13526.00  26788.0   

                           
             75%      max  
감정_대분류                     
기쁨      28602.50  37915.0  
당황      40832.25  51624.0  
분노      40226.25  51626.0  
불안      39126.75  51629.0  
상처      40613.50  51630.0  
슬픔      40091.00  51616.0

In [3]:
import pandas as pd
import numpy as np
file_path = r"C:\Project\Python_Source\AI01\mini_project_09\018.감성대화\Training_221115_add\원천데이터\감성대화말뭉치(최종데이터)_Training.csv"
df = pd.read_csv(file_path)
target_emotions = ['기쁨', '분노', '슬픔', '불안']
df = df[df['감정_대분류'].isin(target_emotions)].copy()
df['사람문장1'] = df['사람문장1'].fillna('')
df['사람문장2'] = df['사람문장2'].fillna('')
df['사람문장3'] = df['사람문장3'].fillna('')
df['total_text'] = df['사람문장1'] + " " + df['사람문장2'] + " " + df['사람문장3']

df = df[["감정_대분류","total_text"]]
df

,감정_대분류,total_text
0,분노,일은 왜 해도 해도 끝이 없을까? 화가 난다. 그냥 내가 해결하는 게 나아. 남들한...
1,분노,이번 달에 또 급여가 깎였어! 물가는 오르는데 월급만 자꾸 깎이니까 너무 화가 나....
2,분노,회사에 신입이 들어왔는데 말투가 거슬려. 그런 애를 매일 봐야 한다고 생각하니까 스...
3,분노,직장에서 막내라는 이유로 나에게만 온갖 심부름을 시켜. 일도 많은 데 정말 분하고 ...
4,분노,얼마 전 입사한 신입사원이 나를 무시하는 것 같아서 너무 화가 나. 상사인 나에게 ...
...,...,...
51621,분노,남편이 내 곁을 떠났어. 아무것도 못 해준 내가 실망스럽고 자꾸 눈물이 나. 내가 ...
51622,분노,건강관리를 너무 안 해서 건강이 좋지 않아 졌어. 주변 사람들에게 폐 끼칠까 봐 걱...
51625,분노,나이가 먹고 이제 돈도 못 벌어 오니까 어떻게 살아가야 할지 막막해. 능력도 없고....
51626,불안,몸이 많이 약해졌나 봐. 이제 전과 같이 일하지 못할 것 같아 너무 짜증 나. 마음...


In [86]:
df.info()

<class 'pandas.DataFrame'>
Index: 24411 entries, 0 to 51625
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   감정_대분류      24411 non-null  str  
 1   total_text  24411 non-null  str  
dtypes: str(2)
memory usage: 6.4 MB


In [4]:
df.groupby("감정_대분류").describe()

total_text                                                            \
            count unique                                                top   
감정_대분류                                                                        
기쁨           6126   6126  퇴사한 지 얼마 안 됐지만 천천히 직장을 구해보려고. 더 좋은 회사가 기다리고 있을...   
분노           9160   9160  일은 왜 해도 해도 끝이 없을까? 화가 난다. 그냥 내가 해결하는 게 나아. 남들한...   
불안           9320   9320  졸업반이라서 취업을 생각해야 하는데 지금 너무 느긋해서 이래도 되나 싶어. 응. 느...   
슬픔           9125   9125  코로나 때문에 뭘 할 수가 없어. 취직 준비를 해야 하는데 시험이 줄줄이 취소되니 ...   

             
       freq  
감정_대분류       
기쁨        1  
분노        1  
불안        1  
슬픔        1

In [5]:
df["감정_대분류"].unique()

<ArrowStringArray>
['분노', '기쁨', '불안', '슬픔']
Length: 4, dtype: str

In [6]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder

# 1. 데이터 로드
file_path = r"C:\Project\Python_Source\AI01\mini_project_09\018.감성대화\Training_221115_add\원천데이터\감성대화말뭉치(최종데이터)_Training.csv"
df = pd.read_csv(file_path)

# 2. 결측치 처리 및 문장 통합
df['사람문장1'] = df['사람문장1'].fillna('')
df['사람문장2'] = df['사람문장2'].fillna('')
df['사람문장3'] = df['사람문장3'].fillna('')
df['total_text'] = df['사람문장1'] + " " + df['사람문장2'] + " " + df['사람문장3']

# 3. 💡 [핵심] 기쁨, 분노, 슬픔 데이터만 필터링
target_emotions = ['기쁨', '분노', '슬픔', '불안']
df_filtered = df[df['감정_대분류'].isin(target_emotions)].copy()

# 4. 필요한 컬럼 추출
data = df_filtered[['total_text', '감정_대분류']].copy()

# 5. 라벨 인코딩
le = LabelEncoder()
data['label'] = le.fit_transform(data['감정_대분류'])
num_labels = len(le.classes_) 

print(f"인코딩된 감정 클래스 목록 (총 {num_labels}개): {le.classes_}")

인코딩된 감정 클래스 목록 (총 4개): ['기쁨' '분노' '불안' '슬픔']


In [7]:
# 1. Validation 데이터 로드
val_file_path = r"C:\Project\Python_Source\AI01\mini_project_09\018.감성대화\Validation_221115_add\원천데이터\감성대화말뭉치(최종데이터)_Validation.csv"
df_val = pd.read_csv(val_file_path, encoding='utf-8-sig')

# 2. 결측치 처리 및 문장 통합
df_val['사람문장1'] = df_val['사람문장1'].fillna('')
df_val['사람문장2'] = df_val['사람문장2'].fillna('')
df_val['사람문장3'] = df_val['사람문장3'].fillna('')
df_val['total_text'] = df_val['사람문장1'] + " " + df_val['사람문장2'] + " " + df_val['사람문장3']

# 3. 💡 [핵심] Validation 데이터도 기쁨, 분노, 슬픔만 필터링
df_val_filtered = df_val[df_val['감정_대분류'].isin(target_emotions)].copy()

# 4. Train에서 선언한 le 객체로 동일하게 변환
df_val_filtered['label'] = le.transform(df_val_filtered['감정_대분류'])

print(f"Validation 데이터 필터링 완료! 클래스 종류: {df_val_filtered['감정_대분류'].unique()}")

Validation 데이터 필터링 완료! 클래스 종류: <ArrowStringArray>
['불안', '슬픔', '기쁨', '분노']
Length: 4, dtype: str


In [14]:
print(le.classes_)

['기쁨' '분노' '불안' '슬픔']


# 모델용 데이터 준비

In [8]:
import torch
from transformers import AutoTokenizer

# 1. 토크나이저 로드
model_name = "beomi/KcELECTRA-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 💡 고속 텍스트 토크나이징 및 Trainer 호환 Dataset 생성 함수
def tokenize_dataset_fast(texts, labels, tokenizer, max_len=64):
    # 전체 텍스트를 한 번에 배치 처리하여 토크나이징 (초고속)
    inputs = tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=max_len,
        return_attention_mask=True,
    )
    
    # Trainer가 요구하는 딕셔너리({key: tensor}) 형태의 리스트 생성
    dataset = []
    for i in range(len(texts)):
        item = {
            'input_ids': torch.tensor(inputs['input_ids'][i], dtype=torch.long),
            'attention_mask': torch.tensor(inputs['attention_mask'][i], dtype=torch.long),
            'labels': torch.tensor(labels[i], dtype=torch.long)
        }
        dataset.append(item)
        
    return dataset

# 2. 데이터셋 생성 (이제 수초 만에 생성되면서 에러도 발생하지 않습니다)
train_dataset = tokenize_dataset_fast(data['total_text'].tolist(), data['label'].tolist(), tokenizer)
val_dataset = tokenize_dataset_fast(df_val_filtered['total_text'].tolist(), df_val_filtered['label'].tolist(), tokenizer)

print(f"데이터셋 변환 완료! (Train: {len(train_dataset)}개, Val: {len(val_dataset)}개)")

데이터셋 변환 완료! (Train: 33731개, Val: 4586개)


# 모델 학습

In [9]:
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

# 분류용 모델 로드 (num_labels는 앞의 셀에서 정의된 값 사용)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# 평가지표 정의 (정확도 및 Macro F1-score)
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {'accuracy': acc, 'macro_f1': f1}

# 학습 하이퍼파라미터 (GPU OOM 방지를 위해 배치 사이즈는 8로 권장)
training_args = TrainingArguments(
    output_dir='./results',          
    num_train_epochs=3,              
    per_device_train_batch_size=8,   
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy="epoch",      # 이전 에러 수정 반영 완료
    save_strategy="epoch",
    load_best_model_at_end=True,   
    fp16=True,
)

# 트레이너 정의 및 학습
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,        # 공식 Validation 데이터셋 투입
    compute_metrics=compute_metrics,
)

# 학습 시작
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base
Key                                               | Status     | Details
--------------------------------------------------+------------+--------
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |        
discriminator_predictions.dense.bias              | UNEXPECTED |        
discriminator_predictions.dense_prediction.weight | UNEXPECTED |        
discriminator_predictions.dense.weight            | UNEXPECTED |        
classifier.out_proj.bias                          | MISSING    |        
classifier.dense.weight                           | MISSING    |        
classifier.out_proj.weight                        | MISSING    |        
classifier.dense.bias                             | MISSING    |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.615437,0.483711,0.833188,0.830759
2,0.511910,0.452555,0.850414,0.848287
3,0.402197,0.475392,0.860009,0.857542


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12651, training_loss=0.5490617452803436, metrics={'train_runtime': 834.3453, 'train_samples_per_second': 121.284, 'train_steps_per_second': 15.163, 'total_flos': 3328184391906816.0, 'train_loss': 0.5490617452803436, 'epoch': 3.0})

In [10]:
# 1. 마지막 체크포인트 경로 설정
last_checkpoint = "./results/checkpoint-12651" # 본인의 결과 폴더에서 마지막 번호 폴더 확인!

# 2. 체크포인트에서 모델 로드
model = AutoModelForSequenceClassification.from_pretrained(last_checkpoint)

# 3. 추가 학습 설정 (7 에포크 추가)
training_args.num_train_epochs = 7  # 추가로 학습할 에포크 수
training_args.output_dir = './results'

# 4. 트레이너 재정의
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# 5. 이어서 학습 시작
trainer.train(resume_from_checkpoint=last_checkpoint)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
4,0.461102,0.513508,0.855648,0.853326
5,0.390171,0.536842,0.872220,0.869435
6,0.260484,0.599428,0.871566,0.869031
7,0.215317,0.647633,0.873746,0.871563


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=29519, training_loss=0.1695880244799861, metrics={'train_runtime': 1094.9066, 'train_samples_per_second': 215.65, 'train_steps_per_second': 26.96, 'total_flos': 7765763581115904.0, 'train_loss': 0.1695880244799861, 'epoch': 7.0})

# 모델 결과 확인

In [7]:
import torch
from sklearn.preprocessing import LabelEncoder
from transformers import AutoModelForSequenceClassification, AutoTokenizer

emotion_classes = ['기쁨', '분노', '슬픔']

le = LabelEncoder()
le.fit(emotion_classes)

checkpoint_path = "./results/checkpoint-9156"
tokenizer = AutoTokenizer.from_pretrained("beomi/kcBERT-LARGE")
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)

print("✅ 토크나이저, 모델, 라벨 인코더 로드 완료!")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✅ 토크나이저, 모델, 라벨 인코더 로드 완료!


In [8]:
import torch
import torch.nn.functional as F

def predict_diary_emotion(text):
    # 1. 모델을 평가 모드로 전환하고 GPU 환경 설정
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    # 2. 입력받은 일기 텍스트 토크나이징
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding="max_length"
    )
    
    # 3. 데이터를 GPU/CPU로 이동
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    
    # 4. 예측 수행 (기울기 계산 비활성화)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        # 확률값으로 변환 (Softmax)
        probabilities = F.softmax(logits, dim=-1).squeeze().cpu().numpy()
        
    # 5. 가장 높은 확률의 인덱스 추출
    pred_class_idx = probabilities.argmax()
    pred_emotion = le.classes_[pred_class_idx]
    pred_confidence = probabilities[pred_class_idx] * 100
    
    # 6. 결과 출력
    print(f"📝 입력한 일기: \"{text}\"")
    print(f"📊 분석 결과: [{pred_emotion}] 일 확률이 {pred_confidence:.2f}%입니다.\n")
    print("[전체 감정별 확률]")
    for emotion, prob in zip(le.classes_, probabilities):
        print(f"- {emotion}: {prob*100:.2f}%")
        
    return pred_emotion

In [9]:
my_diary = "오늘 친구랑 사소한 일로 다... 내 마음을 너무 몰라주는 것 같아서 속상하고 울적한 하루다."
predict_diary_emotion(my_diary)

📝 입력한 일기: "오늘 친구랑 사소한 일로 다... 내 마음을 너무 몰라주는 것 같아서 속상하고 울적한 하루다."
📊 분석 결과: [슬픔] 일 확률이 99.70%입니다.

[전체 감정별 확률]
- 기쁨: 0.03%
- 분노: 0.27%
- 슬픔: 99.70%


np.str_('슬픔')

In [27]:
my_diary2 = "드디어 모델 학습이 끝났다! 에러도 다 해결하고 성능도 잘 나와서 너무 뿌듯하다."
predict_diary_emotion(my_diary2)

📝 입력한 일기: "드디어 모델 학습이 끝났다! 에러도 다 해결하고 성능도 잘 나와서 너무 뿌듯하다."
📊 분석 결과: [기쁨] 일 확률이 99.93%입니다.

[전체 감정별 확률]
- 기쁨: 99.93%
- 분노: 0.05%
- 슬픔: 0.03%


np.str_('기쁨')

In [11]:
predict_diary_emotion("ㅅㅂ 좆같다.")

📝 입력한 일기: "ㅅㅂ 좆같다."
📊 분석 결과: [슬픔] 일 확률이 97.01%입니다.

[전체 감정별 확률]
- 기쁨: 0.08%
- 분노: 2.91%
- 슬픔: 97.01%


np.str_('슬픔')

In [12]:
predict_diary_emotion("ㅅㅂ")

📝 입력한 일기: "ㅅㅂ"
📊 분석 결과: [슬픔] 일 확률이 59.64%입니다.

[전체 감정별 확률]
- 기쁨: 0.58%
- 분노: 39.78%
- 슬픔: 59.64%


np.str_('슬픔')

In [13]:
predict_diary_emotion("아나 힘들구만")

📝 입력한 일기: "아나 힘들구만"
📊 분석 결과: [슬픔] 일 확률이 98.63%입니다.

[전체 감정별 확률]
- 기쁨: 0.05%
- 분노: 1.32%
- 슬픔: 98.63%


np.str_('슬픔')

In [14]:
predict_diary_emotion("잠만 자고 일어나서 오늘 뭐했는지 모르겠다")

📝 입력한 일기: "잠만 자고 일어나서 오늘 뭐했는지 모르겠다"
📊 분석 결과: [슬픔] 일 확률이 99.46%입니다.

[전체 감정별 확률]
- 기쁨: 0.06%
- 분노: 0.48%
- 슬픔: 99.46%


np.str_('슬픔')

In [15]:
predict_diary_emotion("도대체 뭐 하는거야")

📝 입력한 일기: "도대체 뭐 하는거야"
📊 분석 결과: [분노] 일 확률이 83.77%입니다.

[전체 감정별 확률]
- 기쁨: 0.94%
- 분노: 83.77%
- 슬픔: 15.29%


np.str_('분노')

In [26]:
predict_diary_emotion("오늘 상사가 뭐라고 했다. 나쁜놈")

📝 입력한 일기: "오늘 상사가 뭐라고 했다. 나쁜놈"
📊 분석 결과: [분노] 일 확률이 66.01%입니다.

[전체 감정별 확률]
- 기쁨: 1.64%
- 분노: 66.01%
- 슬픔: 32.36%


np.str_('분노')

In [17]:
predict_diary_emotion("뭐라는거야")

📝 입력한 일기: "뭐라는거야"
📊 분석 결과: [분노] 일 확률이 97.93%입니다.

[전체 감정별 확률]
- 기쁨: 0.16%
- 분노: 97.93%
- 슬픔: 1.91%


np.str_('분노')

In [18]:
predict_diary_emotion("멍청이")

📝 입력한 일기: "멍청이"
📊 분석 결과: [분노] 일 확률이 98.18%입니다.

[전체 감정별 확률]
- 기쁨: 0.08%
- 분노: 98.18%
- 슬픔: 1.74%


np.str_('분노')

In [19]:
predict_diary_emotion("비도 오고 삼겹살도 생각나고 친구도 생각나고 그러그러하네")

📝 입력한 일기: "비도 오고 삼겹살도 생각나고 친구도 생각나고 그러그러하네"
📊 분석 결과: [슬픔] 일 확률이 98.29%입니다.

[전체 감정별 확률]
- 기쁨: 0.64%
- 분노: 1.07%
- 슬픔: 98.29%


np.str_('슬픔')

# 감정 분류 및 음악 추천

In [1]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

checkpoint_path = "./results/checkpoint-9156" 

# 💡 모델 구조 자체에 가중치 출력을 활성화(output_attentions=True)하여 로드합니다.
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path, output_attentions=True)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

print("💾 모델 로드 완료!")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

💾 모델 로드 완료!


In [5]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

checkpoint_path = "./results/checkpoint-9156" 

# 💡 모델 구조 자체에 가중치 출력을 활성화(output_attentions=True)하여 로드합니다.
predict_diary_emotion = AutoModelForSequenceClassification.from_pretrained(checkpoint_path, output_attentions=True)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

print("💾 모델 로드 완료!")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

💾 모델 로드 완료!


In [17]:
import torch
import numpy as np

def extract_explainable_keywords(text, top_k=2):
    """
    BERT Attention 가중치를 원래 문장의 띄어쓰기(어절) 단위로 합산하여
    에러나 [UNK] 없이 실제 단어를 100% 추출하는 고도화 함수
    """
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding=False)
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    
    try:
        with torch.no_grad():
            outputs = model(
                input_ids=input_ids, 
                attention_mask=attention_mask, 
                output_attentions=True,
                attn_implementation="eager"
            )
            
        if not hasattr(outputs, 'attentions') or outputs.attentions is None:
            raise ValueError("Attentions is None")
            
        last_layer_attention = outputs.attentions[-1].squeeze(0) 
        cls_attention = last_layer_attention[:, 0, :].mean(dim=0).cpu().numpy()
        tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
        
        # 💡 핵심 고도화: 원래 일기의 단어별로 Attention 점수를 누적할 딕셔너리 생성
        original_words = text.split()
        word_scores = {word: 0.0 for word in original_words}
        
        # 토큰들을 돌면서 원래 단어에 점수 매칭하기
        current_word_idx = 0
        for token, score in zip(tokens, cls_attention):
            if token in ['[CLS]', '[SEP]', '[PAD]']:
                continue
                
            # 토큰 정제 (공백 기호 및 ## 제거)
            clean_token = token.replace(' ', '').replace('##', '')
            if not clean_token:
                continue
                
            # 현재 토큰이 원래 어떤 어절에 포함되는지 매칭하여 점수 합산
            if current_word_idx < len(original_words):
                target_word = original_words[current_word_idx]
                word_scores[target_word] += float(score)
                
                # 토큰이 띄어쓰기 기호( )로 시작하면 다음 단어로 넘어감
                if token.startswith(' '):
                    current_word_idx += 1
                    
        # 점수 순으로 원래 단어 정렬 (의미 없는 짧은 단어 제외)
        sorted_words = sorted(
            [(w, s) for w, s in word_scores.items() if len(w) > 1], 
            key=lambda x: x[1], 
            reverse=True
        )
        
        top_keywords = [word for word, score in sorted_words[:top_k]]
        return top_keywords if top_keywords else original_words[:top_k]

    except Exception as e:
        fallback_words = [w for w in text.split() if len(w) > 1]
        return fallback_words[:top_k] if fallback_words else ["오늘"]

In [19]:
# 테스트용 일기 문장
sample_diary = "오늘 친구랑 사소한 일로 싸웠어. 내 마음을 너무 몰라주는 것 같아서 속상하고 울적한 하루다."

# 1. 기존에 만드신 감정 분석 함수 실행
predicted_emo = predict_diary_emotion(sample_diary)

# 2. 💡 신규 고도화: 감정에 가장 큰 영향을 준 단어 추출
important_words = extract_explainable_keywords(sample_diary, top_k=2)

print("\n==================================================")
print(f"🔍 AI 모델의 판단 근거 (XAI):")
print(f"💡 모델은 일기 속 단어 중 {important_words}에 가장 큰 영향을 받았습니다.")
print("==================================================")

# 3. 이 키워드를 스포티파이 검색 함수에 연동하기
# 예: "속상하고 슬픔 노래" 형태로 검색 쿼리 자동 완성 가능!
chosen_keyword = important_words[0] if important_words else "최신"
print(f"🚀 스포티파이 최종 연동 검색어 예시: '{chosen_keyword} {predicted_emo} 노래'")


🔍 AI 모델의 판단 근거 (XAI):
💡 모델은 일기 속 단어 중 ['오늘', '친구랑']에 가장 큰 영향을 받았습니다.
🚀 스포티파이 최종 연동 검색어 예시: '오늘 분노 노래'


In [23]:
import torch
import torch.nn.functional as F
import numpy as np

def extract_highest_prob_keyword_by_split(text, target_emotion, top_k=2):
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    words = [w.strip(".,!?\"'") for w in text.split()]
    words = [w for w in words if len(w) > 1]
    
    if not words:
        return ["최신"]
        
    # 💡 le.classes_ 대신 우리가 데이터셋에서 정의한 고정 순서 리스트 사용
    # ['기쁨', '분노', '슬픔'] 순서와 일치시킵니다.
    emotion_classes = ['기쁨', '분노', '슬픔']
    try:
        target_idx = emotion_classes.index(target_emotion)
    except ValueError:
        target_idx = 2 # 기본값 슬픔
    
    word_emotion_scores = []
    
    for word in words:
        inputs = tokenizer(word, return_tensors="pt", truncation=True, max_length=32, padding="max_length")
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)
        
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probabilities = F.softmax(logits, dim=-1).squeeze().cpu().numpy()
            
        target_score = probabilities[target_idx]
        word_emotion_scores.append((word, target_score))
        
    word_emotion_scores.sort(key=lambda x: x[1], reverse=True)
    
    print("📊 단어별 감정 확률 스캔 결과:")
    for w, s in word_emotion_scores[:4]:
        print(f"  - [{w}]: 현재 감정 확률 {s*100:.2f}%")
        
    top_keywords = [word for word, score in word_emotion_scores[:top_k]]
    return top_keywords

In [28]:
# 테스트용 일기 문장
sample_diary = "오늘 친구랑 사소한 일로 싸웠어. 내 마음을 너무 몰라주는 것 같아서 속상하고 울적한 하루다."

# 1. 먼저 전체 문장의 감정을 대분류로 파악 (현재 슬픔으로 나오는 상태)
predicted_emo = predict_diary_emotion(sample_diary)

# 2. 💡 유저님의 아이디어: 단어별 쪼개기 매칭 함수 실행
important_words = extract_highest_prob_keyword_by_split(sample_diary, target_emotion=predicted_emo, top_k=2)

print("\n==================================================")
print(f"🔍 AI 모델의 판단 근거 (단어 스캔 방식):")
print(f"💡 모델이 보기에 [{predicted_emo}] 감정이 가장 진하게 묻어나는 단어 Top 2는 {important_words} 입니다.")
print("==================================================")

# 3. 스포티파이 검색어 연동
if important_words:
    raw_keyword = important_words[0]
    # 우리가 만든 변환 함수를 거치게 합니다!
    chosen_keyword = convert_to_search_keyword(raw_keyword) 
else:
    chosen_keyword = "최신"

print(f"🚀 스포티파이 최종 연동 검색어 예시: '{chosen_keyword} {predicted_emo} 노래'")

📊 원래 문장의 [분노] 확률: 69.09%
📉 단어 제거 시 감정 확률 하락도 스캔 결과:
  - [싸웠어] 제외 시: 확률 -13.33% 변화
  - [하루다] 제외 시: 확률 -13.33% 변화
  - [오늘] 제외 시: 확률 -12.86% 변화
  - [친구랑] 제외 시: 확률 -12.86% 변화

🔍 AI 모델의 판단 근거 (단어 스캔 방식):
💡 모델이 보기에 [분노] 감정이 가장 진하게 묻어나는 단어 Top 2는 ['싸웠어', '하루다'] 입니다.
🚀 스포티파이 최종 연동 검색어 예시: '싸움 분노 노래'


In [25]:
import torch
import torch.nn.functional as F
import numpy as np

def extract_highest_prob_keyword_by_split(text, target_emotion, top_k=2):
    """
    [패딩 확률 고정 오류 해결 버전]
    문장에서 각 단어를 하나씩 제외(Drop)해 보며, 
    해당 감정 확률을 가장 크게 떨어뜨리는 '핵심 감정 키워드'를 역추적합니다.
    """
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    original_words = text.split()
    # 2글자 이상의 유효한 단어 필터링
    valid_words = [w.strip(".,!?\"'") for w in original_words]
    valid_words = [w for w in valid_words if len(w) > 1]
    
    if not valid_words:
        return ["최신"]
        
    # 1. 원래 전체 문장의 기준 확률(%) 계산
    inputs_orig = tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding="max_length")
    with torch.no_grad():
        outputs_orig = model(input_ids=inputs_orig["input_ids"].to(device), attention_mask=inputs_orig["attention_mask"].to(device))
        probs_orig = F.softmax(outputs_orig.logits, dim=-1).squeeze().cpu().numpy()
    
    emotion_classes = ['기쁨', '분노', '슬픔']
    target_idx = emotion_classes.index(target_emotion)
    orig_target_score = probs_orig[target_idx] # 예: 원래 문장의 슬픔 확률 (99%)
    
    word_importance_scores = []
    
    # 2. 단어를 하나씩 빼보며 확률 변화 관찰
    for word in valid_words:
        # 현재 단어만 제외한 나머지 단어들로 가상 문장 조합
        modified_words = [w for w in original_words if w.strip(".,!?\"'") != word]
        modified_text = " ".join(modified_words)
        
        # 모델에 입력
        inputs_mod = tokenizer(modified_text, return_tensors="pt", truncation=True, max_length=128, padding="max_length")
        with torch.no_grad():
            outputs_mod = model(input_ids=inputs_mod["input_ids"].to(device), attention_mask=inputs_mod["attention_mask"].to(device))
            probs_mod = F.softmax(outputs_mod.logits, dim=-1).squeeze().cpu().numpy()
            
        mod_target_score = probs_mod[target_idx]
        
        # 원래 확률에서 이 단어를 뺐을 때 감소한 양 (감소 폭이 클수록 중요한 감정 단어!)
        drop_impact = orig_target_score - mod_target_score
        word_importance_scores.append((word, drop_impact))
        
    # 3. 영향력(감소 폭)이 큰 순서대로 정렬
    word_importance_scores.sort(key=lambda x: x[1], reverse=True)
    
    print(f"📊 원래 문장의 [{target_emotion}] 확률: {orig_target_score*100:.2f}%")
    print("📉 단어 제거 시 감정 확률 하락도 스캔 결과:")
    for w, score in word_importance_scores[:4]:
        print(f"  - [{w}] 제외 시: 확률 {-score*100:+.2f}% 변화")
        
    top_keywords = [word for word, score in word_importance_scores[:top_k]]
    return top_keywords

In [27]:
from konlpy.tag import Okt
okt = Okt()

def convert_to_search_keyword(word):
    """
    '싸웠어' -> '싸움' (명사형) 또는 '싸우다' (기본형)
    '속상하고' -> '속상하다' (기본형) 형태로 변환해주는 함수
    """
    # 특수문자나 조사 흔적을 먼저 깔끔하게 지웁니다.
    clean_word = word.strip(".,!?\"'")
    
    # Okt를 이용해 품사 분석을 수행합니다.
    pos_tags = okt.pos(clean_word, stem=True) # stem=True를 주면 '싸웠어'가 '싸우다'로 기본형 변환됩니다.
    
    if not pos_tags:
        return clean_word
        
    # 예: '싸웠어' 분석 시 -> [('싸우다', 'Verb')]
    base_word, pos = pos_tags[0]
    
    # 🎯 [고도화 규칙] 동사이고 '싸우다' 계열이면 명사형인 '싸움'으로 직접 매핑
    if pos == 'Verb' and base_word == '싸우다':
        return "싸움"
    
    # 그 외의 형용사나 동사(예: '속상하고' -> '속상하다')는 깔끔하게 기본형태로 반환
    if pos in ['Verb', 'Adjective']:
        return base_word
        
    return clean_word

In [38]:
import torch
import torch.nn.functional as F
import numpy as np
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from konlpy.tag import Okt

okt = Okt()

def convert_to_search_keyword(word):
    """단어를 형태소 분석기로 정제"""
    clean_word = word.strip(".,!?\"'")
    pos_tags = okt.pos(clean_word, stem=True) 
    if not pos_tags: return clean_word
    base_word, pos = pos_tags[0]
    if pos == 'Verb' and base_word == '싸우다': return "싸움"
    if pos in ['Verb', 'Adjective']: return base_word
    return clean_word


def recommend_music_by_integrated_pipeline(diary_text, client_secret):
    """
    [감정 고정 오류 완벽 해결 버전]
    유저님의 le.classes_를 직접 참조하여 클래스 매칭 오류를 근본적으로 해결합니다.
    """
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    print("🧠 1. 일기 감정 분석 중...")
    # ----------------------------------------------------
    # [1단계] BERT 전체 문장 감정 분류 (le.classes_ 기반)
    # ----------------------------------------------------
    inputs_orig = tokenizer(diary_text, return_tensors="pt", truncation=True, max_length=128, padding="max_length")
    with torch.no_grad():
        outputs_orig = model(input_ids=inputs_orig["input_ids"].to(device), attention_mask=inputs_orig["attention_mask"].to(device))
        probs_orig = F.softmax(outputs_orig.logits, dim=-1).squeeze().cpu().numpy()
    
    # 💡 [핵심 수정] 임의의 리스트 대신, 노트북 상단에 살아있는 진짜 le.classes_를 직접 사용합니다!
    final_target_idx = probs_orig.argmax() 
    detected_emotion = le.classes_[final_target_idx] 
    orig_target_score = probs_orig[final_target_idx]
    
    print(f"   ▶ 분석 결과: [{detected_emotion}] (확률: {orig_target_score*100:.2f}%)")
    
    print("\n🔎 2. 감정 유발 핵심 키워드 추적 중 (단어 제거 방식)...")
    # ----------------------------------------------------
    # [2단계] 단어 제거 루프 (위에서 고정한 final_target_idx 유지)
    # ----------------------------------------------------
    original_words = diary_text.split()
    valid_words = [w.strip(".,!?\"'") for w in original_words if len(w.strip(".,!?\"'")) > 1]
    
    chosen_keyword = "최신"
    if valid_words:
        word_importance_scores = []
        for word in valid_words:
            modified_words = [w for w in original_words if w.strip(".,!?\"'") != word]
            modified_text = " ".join(modified_words)
            
            if not modified_text.strip():
                continue
                
            inputs_mod = tokenizer(modified_text, return_tensors="pt", truncation=True, max_length=128, padding="max_length")
            with torch.no_grad():
                outputs_mod = model(input_ids=inputs_mod["input_ids"].to(device), attention_mask=inputs_mod["attention_mask"].to(device))
                probs_mod = F.softmax(outputs_mod.logits, dim=-1).squeeze().cpu().numpy()
                
            # 💡 1차 전체 분류에서 얻은 진짜 타겟 감정 인덱스만 추적하도록 고정
            mod_target_score = probs_mod[final_target_idx]
            drop_impact = orig_target_score - mod_target_score
            word_importance_scores.append((word, drop_impact))
            
        if word_importance_scores:
            word_importance_scores.sort(key=lambda x: x[1], reverse=True)
            raw_important_word = word_importance_scores[0][0]
            print(f"   ▶ 원본 핵심 단어 추출 완료: '{raw_important_word}'")
            
            # [3단계] 원형 정제
            chosen_keyword = convert_to_search_keyword(raw_important_word)
            print(f"   ▶ 형태소 원형 변환 완료: '{raw_important_word}' ➡️ '{chosen_keyword}'")
    
    chosen_query = f"{chosen_keyword} {detected_emotion} 노래"
    print(f"🎯 4. 최종 스포티파이 검색 쿼리: '{chosen_query}'")
    
    # ----------------------------------------------------
    # [4단계] 스포티파이 API 연동 및 곡 검색 추천
    # ----------------------------------------------------
    try:
        client_id = "d479ee2e82c14d7eaea40dbe432663d1" 
        auth_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
        sp = spotipy.Spotify(auth_manager=auth_manager)
        
        search_results = sp.search(q=chosen_query, type="track", limit=5, market="KR")
        tracks = search_results.get("tracks", {}).get("items", [])
        
        if not tracks:
            print(f"⚠️ '{chosen_query}' 결과가 없어 기본 감정어로 재검색합니다.")
            search_results = sp.search(q=f"{detected_emotion} 노래", type="track", limit=5, market="KR")
            tracks = search_results.get("tracks", {}).get("items", [])
            
        print(f"\n🎵 [오늘의 일기 본문 맞춤 음악 추천 리스트]")
        print("-" * 70)
        
        dashboard_data_list = []
        for idx, track in enumerate(tracks, 1):
            track_name = track.get("name")
            artists = ", ".join([a["name"] for a in track.get("artists", [])])
            images = track.get("album", {}).get("images", [])
            image_url = images[0].get("url") if images else "https://via.placeholder.com/300"
            preview_url = track.get("preview_url")
            
            print(f"{idx}. {track_name} - {artists}")
            
            dashboard_data_list.append({
                "rank": idx, "title": track_name, "artist": artists, 
                "image": image_url, "preview": preview_url
            })
        return dashboard_data_list
        
    except Exception as e:
        print(f"❌ 스포티파이 연동 중 오류 발생: {e}")
        return []

In [39]:

# 2. 테스트용 일기 본문
my_diary = "오늘 친구랑 사소한 일로 싸웠어. 내 마음을 너무 몰라주는 것 같아서 속상하고 울적한 하루다."

# 3. 통합 함수 원격 호출!
result_dashboard = recommend_music_by_integrated_pipeline(my_diary, MY_SECRET_KEY)

🧠 1. 일기 감정 분석 중...
   ▶ 분석 결과: [분노] (확률: 69.09%)

🔎 2. 감정 유발 핵심 키워드 추적 중 (단어 제거 방식)...
   ▶ 원본 핵심 단어 추출 완료: '싸웠어'
   ▶ 형태소 원형 변환 완료: '싸웠어' ➡️ '싸움'
🎯 4. 최종 스포티파이 검색 쿼리: '싸움 분노 노래'

🎵 [오늘의 일기 본문 맞춤 음악 추천 리스트]
----------------------------------------------------------------------
1. 승리를 위하여 - TransFixion
2. 싸운날 - BOL4
3. 220714 - Huckleberry P
4. 축제 - Baek Z Young
5. House Music - Desos


In [52]:
test_diaries = [
    "오늘 프로젝트 마일스톤 결과도 잘 나오고 조원들이랑 삼겹살 먹기로 해서 너무 신난다!",
    "밤늦게 혼자 한강 공원을 걷는데 문득 옛날 생각이 나면서 눈물이 나고 슬펐다.",
    "팀원이 약속한 마감 시간을 또 어겨서 진짜 짜증 나고 머리끝까지 화가 치밀어 오른다."
]

for text in test_diaries:
    result_dashboard = recommend_music_by_integrated_pipeline(text, MY_SECRET_KEY)

🧠 1. 일기 감정 분석 중...
   ▶ 분석 결과: [기쁨]

🔎 2. 감정 유발 핵심 키워드 추적 중 (단어 제거 방식)...
   ▶ 원본 핵심 단어 추출 완료: '오늘'
   ▶ 형태소 원형 변환 완료: '오늘' ➡️ '오늘'
🎯 4. 최종 스포티파이 검색 쿼리: '오늘 기쁨 노래'

🎵 [오늘의 일기 본문 맞춤 음악 추천 리스트]
----------------------------------------------------------------------
1. 기쁨의 날 주시네 Your Given Day - Markers Worship
2. You and I (Park Bom) - 2NE1
3. 기쁨의 노래 - 숲속의 피아노
4. IF I SAY, I LOVE YOU - BOYNEXTDOOR
5. 쉬어가자 (Pause Road) - 제이엠진 JM jin
🧠 1. 일기 감정 분석 중...
   ▶ 분석 결과: [슬픔]

🔎 2. 감정 유발 핵심 키워드 추적 중 (단어 제거 방식)...
   ▶ 원본 핵심 단어 추출 완료: '슬펐다'
   ▶ 형태소 원형 변환 완료: '슬펐다' ➡️ '슬프다'
🎯 4. 최종 스포티파이 검색 쿼리: '슬프다 슬픔 노래'

🎵 [오늘의 일기 본문 맞춤 음악 추천 리스트]
----------------------------------------------------------------------
1. 슬픈 인연 - 015B
2. 이미 슬픈 사랑 - 야다
3. Fly Up - Lookism
4. 슬프다 슬퍼 Sad, Oh Sad - The Freaks
5. Circles - SEVENTEEN
🧠 1. 일기 감정 분석 중...
   ▶ 분석 결과: [분노]

🔎 2. 감정 유발 핵심 키워드 추적 중 (단어 제거 방식)...
   ▶ 원본 핵심 단어 추출 완료: '팀원이'
   ▶ 형태소 원형 변환 완료: '팀원이' ➡️ '팀원이'
🎯 4. 최종 스포티파이 검색 쿼리: '팀원이 분노 노래'

🎵 [오늘의 일기 본문 맞춤 음악 추

In [43]:
import torch
import torch.nn.functional as F
import numpy as np
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from konlpy.tag import Okt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import random

okt = Okt()

def convert_to_noun_form(word):
    """
    '모르다' -> '모름', '신나다' -> '신남', '먹다' -> '먹음' 등
    동사/형용사 원형을 자연스러운 명사형으로 변환합니다.
    """
    if not word.endswith('다') or len(word) <= 1:
        return word # '비', '기분' 같은 명사는 그대로 통과
        
    stem = word[:-1]       # '다'를 뗀 앞부분 (예: '모르', '먹')
    last_char = stem[-1]   # 마지막 글자 (예: '르', '먹')
    
    # 한글 유니코드 계산을 통해 받침 유무 확인
    char_code = ord(last_char) - 0xAC00
    if char_code < 0 or char_code > 11171:
        return word
        
    jongseong = char_code % 28 # 종성(받침) 위치 확인
    
    # 1. 받침이 없는 경우 (예: 모르 -> 모름, 신나 -> 신남)
    if jongseong == 0:
        new_char = chr(ord(last_char) + 16) # 'ㅁ' 받침 추가
        return stem[:-1] + new_char
        
    # 2. 받침이 'ㄹ'인 경우 (예: 살다 -> 삶, 만들다 -> 만듦)
    elif jongseong == 8:
        new_char = chr(ord(last_char) + 2)  # 'ㄻ' 받침으로 변경
        return stem[:-1] + new_char
        
    # 3. 그 외 받침이 이미 있는 경우 (예: 먹다 -> 먹음, 잡다 -> 잡음)
    else:
        return stem + "음"

def convert_to_search_keyword(word):
    clean_word = word.strip(".,!?\"'")
    
    # 1. 의미 없는 부사 및 감탄사 블랙리스트 방어
    blacklist_words = ["도대체", "진짜", "정말", "너무", "완전", "그냥", "막상", "어차피", "갑자기"]
    if clean_word in blacklist_words:
        return ""
        
    # stem=True로 기본 원형 복원 진행
    pos_tags = okt.pos(clean_word, stem=True) 
    if not pos_tags: 
        return clean_word
        
    # 2. 사용할 핵심 품사(명사, 동사, 형용사)만 허용하는 화이트리스트 방식 적용
    valid_morphemes = [
        (text, pos) for text, pos in pos_tags 
        if pos in ['Noun', 'Verb', 'Adjective']
    ]
    
    if not valid_morphemes:
        return ""
        
    # 가장 핵심이 되는 첫 번째 형태소의 단어와 품사 추출
    base_word, pos = valid_morphemes[0]
    
    # 3. 한 글자짜리 동사/형용사나 의미 없는 단어 추가 필터링
    # 예: '해서' -> '하다'(Verb)인데 '하다' 자체는 검색어로 무의미하므로 제외
    if pos in ['Verb', 'Adjective'] and (len(base_word) <= 1 or base_word == '하다'):
        return ""
        
    return base_word


def recommend_music_by_integrated_pipeline(diary_text, client_secret, model_path="./results/checkpoint-12651"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    tokenizer = AutoTokenizer.from_pretrained("beomi/KcELECTRA-base") 
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.eval()
    model.to(device)
    
    # 감정 라벨 순서 추출
    if hasattr(model.config, "id2label") and "LABEL" not in str(model.config.id2label):
        emotion_list = [model.config.id2label[i] for i in range(len(model.config.id2label))]
    else:
        emotion_list = ["기쁨", "분노", "불안", "슬픔"] # 기본 fallback 순서
    
    # 🔍 1단계: 감정 분석 및 디버깅 로그 출력
    print("원 문장:",diary_text)
    print("🧠 1. 일기 감정 분석 중...")
    inputs_orig = tokenizer(diary_text, return_tensors="pt", truncation=True, max_length=128)
    
    with torch.no_grad():
        outputs_orig = model(
            input_ids=inputs_orig["input_ids"].to(device), 
            attention_mask=inputs_orig["attention_mask"].to(device)
        )
        probs_orig = F.softmax(outputs_orig.logits, dim=-1).squeeze().cpu().numpy()
    
    final_target_idx = probs_orig.argmax() 
    detected_emotion = emotion_list[final_target_idx] 
    orig_target_score = probs_orig[final_target_idx]
    
    # 🚨 [핵심 디버그 프린트] 이 세 줄의 출력을 확인해야 합니다!
    print(f"   ⚠️ [디버그] 현재 인식된 감정 순서 리스트: {emotion_list}")
    print(f"   ⚠️ [디버그] 모델의 생짜 확률 배열: {probs_orig}")
    print(f"   ⚠️ [디버그] 선택된 가장 높은 인덱스(argmax): {final_target_idx}")
    print(f"   ▶ 최종 매칭 결과: [{detected_emotion}] (확률: {orig_target_score*100:.2f}%)")
    
    # 🔍 2단계: 키워드 추적 (조사/어미 분리 오류 완벽 해결 버전)
    print("\n🔎 2. 감정 유발 핵심 키워드 추적 중 (형태소 단위 정밀 분석)...")
    stopwords = [
        "오늘", "군은", "오늘은", "오늘도", "어제", "내일", "모레", "요즘", "요즘은", "최근", "그때", 
        "지금", "이제", "현재", "아까", "방금", "벌써", "맨날", "항상", "자주", "가끔", 
        "평소", "매일", "하루", "갑자기", "드디어", "결국", "마침내", "올해", "이번", "지난", 
        "동안", "순간", "아침", "점심", "저녁", "새벽", "주말", "하루종일", "조원", "팀원", "조원들이랑",
        "그리고", "하지만", "그러나", "그래서", "그래도", "그런데", "그렇지만", "그러면", 
        "그러니까", "아무튼", "어쨌든", "일단", "역시", "참", "또", "더", "다시", "그냥", 
        "너무", "너무나", "진짜", "진짜로", "정말", "정말로", "매우", "아주", "무척", "엄청", 
        "상당히", "조금", "약간", "되게", "꽤", "제일", "가장", "훨씬", "많이", "완전", "완전히",
        "나", "나는", "내가", "나를", "우리", "우리는", "우리끼리", "너", "너는",
        "생각", "생각이", "생각을", "소리", "얘기", "이야기", "때문", "때문에", "덕분에",
        "도대체", "막상", "어차피", "만큼", "정도", "경우", "때문", "때문에", "덕분에", "무엇", "무슨", "가지", "번은",
        "모르다", "그렇다", "아니다", "없다", "있다", "같다", "되다", "하다", "보이다", "내리다", '내림', 
        "결과", "시험", "과제", "마일스톤", "이유", "사실", "상황", "문제", "내용", 
        "행동", "부분", "모습", "소식", "시간", "약속", "시작", "마지막", "처음", "노력",
        "일어나다", "자다", "먹다", "가다", "오다", "하다", "선택", "일어나", "일어나서"
    ]
    
    # 문장 전체의 원본 형태소와 원형(Stem) 형태소를 문맥에 맞게 각각 추출합니다.
    morphemes_orig = okt.pos(diary_text, stem=False)
    morphemes_stem = okt.pos(diary_text, stem=True)
    
    # 중요도 테스트를 진행할 후보 단어(원형) 리스트업
    candidate_stems = []
    for i in range(len(morphemes_stem)):
        text_stem, pos = morphemes_stem[i]
        # 명사, 동사, 형용사만 후보로 선정 (의미 없는 단어 및 조사/어미 원천 차단)
        if pos in ['Noun', 'Verb', 'Adjective']:
            if len(text_stem) > 1 and text_stem not in stopwords and text_stem != "하다":
                if text_stem not in candidate_stems:
                    candidate_stems.append(text_stem)
                    
    chosen_keyword = ""
    if candidate_stems:
        word_importance_scores = []
        for target_stem in candidate_stems:
            # 형태소 단위로 해당 단어를 정밀하게 제외한 modified_text 생성
            modified_tokens = []
            for i in range(len(morphemes_stem)):
                orig_text, _ = morphemes_orig[i]
                stem_text, _ = morphemes_stem[i]
                
                # 현재 검사 중인 단어의 원형과 일치하면 문장에서 완전히 제외
                if stem_text == target_stem:
                    continue
                modified_tokens.append(orig_text)
                
            modified_text = " ".join(modified_tokens)
            if not modified_text.strip(): continue
                
            # 모델 예측값 변화 측정 (Occlusion Test)
            inputs_mod = tokenizer(modified_text, return_tensors="pt", truncation=True, max_length=128)
            with torch.no_grad():
                outputs_mod = model(
                    input_ids=inputs_mod["input_ids"].to(device), 
                    attention_mask=inputs_mod["attention_mask"].to(device)
                )
                probs_mod = F.softmax(outputs_mod.logits, dim=-1).squeeze().cpu().numpy()
            
            mod_target_score = probs_mod[final_target_idx]
            word_importance_scores.append((target_stem, -mod_target_score))
            
        # ... [기존 2단계 루프 맨 하단 점수 정렬 직후 부분] ...
        if word_importance_scores:
            word_importance_scores.sort(key=lambda x: x[1], reverse=True)
            
            chosen_keyword = "" # 기본값
            
            # ↙️ 중요도 순으로 정렬된 단어들을 하나씩 검사하며 최적의 단어 탐색
            for item in word_importance_scores:
                target_stem = item[0] # 원형 단어 (예: '모르다', '오다', '짜증')
                converted = convert_to_noun_form(target_stem) # 명사화 (예: '모름', '옴', '짜증')
                
                # 🚫 [필터링 규칙 1] 원래 동사/형용사였는데 명사화 후 1글자가 되는 단어 및 일상 동사 원형 패스
                # 💡 target_stem 비교군에 "나오다", "자다"를 확실하게 추가합니다.
                if target_stem in ["오다", "나다", "가다", "자다", "보다", "이다", "일어나다", "일어나", "어떻다", "나오다", "자고"] or len(converted) <= 1:
                    if target_stem not in ["비", "화", "돈"]:
                        continue
                
                # 🚫 [필터링 규칙 2] 구어체 찌꺼기 및 어색한 변환 명사 패스
                # 💡 converted 비교군에 "자고 나옴", "나옴"을 추가합니다.
                if converted in ["뭐람", "그렇다", "어쩌다", "하다", "일어남", "일어나서", "어떻음", "자고 나옴", "나옴"]:
                    continue
                            
                # ✨ 위 필터링을 다 통과한 가장 자연스러운 단어
                chosen_keyword = converted
                
                # 특정 필수 예외 처리만 살짝 적용
                if chosen_keyword == "비도": 
                    chosen_keyword = "비"
                if chosen_keyword == "화가": 
                    chosen_keyword = "화"
                
                break # 최적의 단어를 찾았으므로 루프 탈출
                   
    print(f"   ▶ 최종 핵심 단어 선정: '{chosen_keyword}'")
    
    # 💡 [예외 처리] 감정과 키워드가 겹치는지 체크하는 로직
    # 감정 리스트 예시: ["기쁨", "분노", "불안", "슬픔"]
    # 키워드가 이미 감정명과 비슷하다면 키워드만 사용하거나 감정을 우선시함
    
    # 중복 의미 매핑 (검색 품질 향상)
    overlap_map = {
        "기쁨": ["기쁨", "즐거움", "신남", "행복"],
        "분노": ["분노", "화", "짜증", "분함"],
        "불안": ["불안", "걱정", "초조"],
        "슬픔": ["슬픔", "우울", "눈물"]
    }
    
    # 감정이 키워드에 포함되어 있는지 확인
    is_overlapping = False
    for label, variations in overlap_map.items():
        if detected_emotion == label and chosen_keyword in variations:
            is_overlapping = True
            break
            
    # 🎯 3. 최종 스포티파이 검색 쿼리 결정 (예외 처리 적용)
    if is_overlapping:
        # 중복 시 감정 태그만 사용 (더 넓은 범위의 음악 검색 가능)
        chosen_query = f"#{detected_emotion}"
    elif chosen_keyword == "":
        chosen_query = f"#{detected_emotion}"
    else:
        # 겹치지 않으면 키워드 + 감정 조합
        chosen_query = f"#{chosen_keyword} #{detected_emotion}"
        
    print(f"🎯 3. 최종 스포티파이 검색 쿼리: '{chosen_query}'")
    
    # 3단계: 스포티파이 검색
    try:
        # 오프셋 범위를 100으로 넓혀 랜덤성 극대화
        random_offset = random.randint(0, 100) 
        client_id = "d479ee2e82c14d7eaea40dbe432663d1" 
        auth_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
        sp = spotipy.Spotify(auth_manager=auth_manager)
        
        # limit=1로 설정하여 결과 딱 1개만 가져옴
        search_results = sp.search(q=chosen_query, type="track", limit=1, offset=random_offset)
        tracks = search_results.get("tracks", {}).get("items", [])
        
        # 결과가 없으면 태그 없이 감정으로만 재검색
        if not tracks:
            search_results = sp.search(q=f"#{detected_emotion}", type="track", limit=1, offset=random_offset)
            tracks = search_results.get("tracks", {}).get("items", [])
            
        print(f"\n🎵 [오늘의 일기 본문 맞춤 추천 음악]")
        print("-" * 70)
        
        dashboard_data_list = []
        for track in tracks:
            track_name = track.get("name")
            artists = ", ".join([a["name"] for a in track.get("artists", [])])
            images = track.get("album", {}).get("images", [])
            image_url = images[0].get("url") if images else "https://via.placeholder.com/300"
            preview_url = track.get("preview_url")
            
            print(f"추천곡: {track_name} - {artists}")
            dashboard_data_list.append({
                "rank": 1, 
                "title": track_name, 
                "artist": artists, 
                "image": image_url, 
                "preview": preview_url
            })
            
        return dashboard_data_list

    except Exception as e:
        print(f"❌ 스포티파이 연동 중 오류 발생: {e}")
        return []

In [44]:
from config import SPOTIFY_SECRET_KEY

MY_SECRET_KEY = SPOTIFY_SECRET_KEY

In [45]:
test_diaries = [
    "오늘 프로젝트 마일스톤 결과도 잘 나오고 조원들이랑 삼겹살 먹기로 해서 너무 신난다!",
    "밤늦게 혼자 한강 공원을 걷는데 문득 옛날 생각이 나면서 눈물이 나고 슬펐다.",
    "팀원이 약속한 마감 시간을 또 어겨서 진짜 짜증 나고 머리끝까지 화가 치밀어 오른다.",
    "오늘 친구랑 사소한 일로 싸웠어. 내 마음을 너무 몰라주는 것 같아서 속상하고 울적한 하루다.",
    "오늘 미니 프로젝트 결과가 너무 잘 나와서 기분이 날아갈 것 같아!",
    "비도 오고 삼겹살도 생각나고 친구도 생각나고 그러그러하네",
    "오늘 상사가 뭐라고 했다. 나쁜",
    "도대체 뭐 하는거야",
    "잠만 자고 일어나서 오늘 뭐했는지 모르겠다",
    "오랜만에 친구들을 만나서 맛있는 것도 먹고 수다 떨었더니 너무 신난다!",
    "노력한 만큼 결과가 안 나와서 속상하다. 앞으로 어떻게 해야 할지 잘 모르겠다.",
    "도대체 왜 나한테만 이런 일이 일어나는지 진짜 짜증나고 화가 치밀어 난다.",
    "오늘도 창밖에 비도 내리고 기분도 꿀꿀해서 센치해지는 밤이다.",
    "팀 프로젝트 마감이 코앞인데 다들 참여를 안 해서 답답하고 스트레스 받는다."
]

for text in test_diaries:
    result_dashboard = recommend_music_by_integrated_pipeline(text, MY_SECRET_KEY)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

원 문장: 오늘 프로젝트 마일스톤 결과도 잘 나오고 조원들이랑 삼겹살 먹기로 해서 너무 신난다!
🧠 1. 일기 감정 분석 중...
   ⚠️ [디버그] 현재 인식된 감정 순서 리스트: ['기쁨', '분노', '불안', '슬픔']
   ⚠️ [디버그] 모델의 생짜 확률 배열: [9.9842775e-01 4.3066018e-04 6.6588761e-04 4.7572597e-04]
   ⚠️ [디버그] 선택된 가장 높은 인덱스(argmax): 0
   ▶ 최종 매칭 결과: [기쁨] (확률: 99.84%)

🔎 2. 감정 유발 핵심 키워드 추적 중 (형태소 단위 정밀 분석)...
   ▶ 최종 핵심 단어 선정: '신남'
🎯 3. 최종 스포티파이 검색 쿼리: '#기쁨'

🎵 [오늘의 일기 본문 맞춤 추천 음악]
----------------------------------------------------------------------
추천곡: 기쁨의 단 Songs of Joy - 김단비 Kim Dan Bi


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

원 문장: 밤늦게 혼자 한강 공원을 걷는데 문득 옛날 생각이 나면서 눈물이 나고 슬펐다.
🧠 1. 일기 감정 분석 중...
   ⚠️ [디버그] 현재 인식된 감정 순서 리스트: ['기쁨', '분노', '불안', '슬픔']
   ⚠️ [디버그] 모델의 생짜 확률 배열: [0.00111768 0.00553519 0.00940647 0.98394066]
   ⚠️ [디버그] 선택된 가장 높은 인덱스(argmax): 3
   ▶ 최종 매칭 결과: [슬픔] (확률: 98.39%)

🔎 2. 감정 유발 핵심 키워드 추적 중 (형태소 단위 정밀 분석)...
   ▶ 최종 핵심 단어 선정: '슬픔'
🎯 3. 최종 스포티파이 검색 쿼리: '#슬픔'

🎵 [오늘의 일기 본문 맞춤 추천 음악]
----------------------------------------------------------------------
추천곡: 눈물아 슬픔아 - SOYA


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

원 문장: 도대체 왜 나한테만 이런 일이 일어나는지 진짜 짜증나고 화가 치밀어 난다.
🧠 1. 일기 감정 분석 중...
   ⚠️ [디버그] 현재 인식된 감정 순서 리스트: ['기쁨', '분노', '불안', '슬픔']
   ⚠️ [디버그] 모델의 생짜 확률 배열: [0.00127154 0.98298764 0.0081728  0.00756804]
   ⚠️ [디버그] 선택된 가장 높은 인덱스(argmax): 1
   ▶ 최종 매칭 결과: [분노] (확률: 98.30%)

🔎 2. 감정 유발 핵심 키워드 추적 중 (형태소 단위 정밀 분석)...
   ▶ 최종 핵심 단어 선정: '화'
🎯 3. 최종 스포티파이 검색 쿼리: '#분노'

🎵 [오늘의 일기 본문 맞춤 추천 음악]
----------------------------------------------------------------------
추천곡: I Won - Ty Dolla $ign, Jack Harlow, 24kGoldn


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

원 문장: 오늘도 창밖에 비도 내리고 기분도 꿀꿀해서 센치해지는 밤이다.
🧠 1. 일기 감정 분석 중...
   ⚠️ [디버그] 현재 인식된 감정 순서 리스트: ['기쁨', '분노', '불안', '슬픔']
   ⚠️ [디버그] 모델의 생짜 확률 배열: [0.13503426 0.18590716 0.4131826  0.26587594]
   ⚠️ [디버그] 선택된 가장 높은 인덱스(argmax): 2
   ▶ 최종 매칭 결과: [불안] (확률: 41.32%)

🔎 2. 감정 유발 핵심 키워드 추적 중 (형태소 단위 정밀 분석)...
   ▶ 최종 핵심 단어 선정: '센치함'
🎯 3. 최종 스포티파이 검색 쿼리: '#센치함 #불안'

🎵 [오늘의 일기 본문 맞춤 추천 음악]
----------------------------------------------------------------------
추천곡: 고요 - 바람새


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

원 문장: 팀 프로젝트 마감이 코앞인데 다들 참여를 안 해서 답답하고 스트레스 받는다.
🧠 1. 일기 감정 분석 중...
   ⚠️ [디버그] 현재 인식된 감정 순서 리스트: ['기쁨', '분노', '불안', '슬픔']
   ⚠️ [디버그] 모델의 생짜 확률 배열: [0.00116957 0.02013768 0.96876913 0.00992367]
   ⚠️ [디버그] 선택된 가장 높은 인덱스(argmax): 2
   ▶ 최종 매칭 결과: [불안] (확률: 96.88%)

🔎 2. 감정 유발 핵심 키워드 추적 중 (형태소 단위 정밀 분석)...
   ▶ 최종 핵심 단어 선정: '스트레스'
🎯 3. 최종 스포티파이 검색 쿼리: '#스트레스 #불안'

🎵 [오늘의 일기 본문 맞춤 추천 음악]
----------------------------------------------------------------------
추천곡: 대나무 피리 - 스트레스 해소 친구


In [11]:
import torch
from transformers import pipeline, AutoModelForCausalLM

MODEL = 'beomi/KoAlpaca-Polyglot-5.8B'

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
).to(device=f"cuda", non_blocking=True)
model.eval()

pipe = pipeline(
    'text-generation', 
    model=model,
    tokenizer=MODEL,
    device=0
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

C:\Project\Python_Source\AI01\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Win11Pro\.cache\huggingface\hub\models--beomi--KoAlpaca-Polyglot-5.8B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors.index.json:   0%|          | 0.00/36.8k [00:00<?, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'temperature', 'max_new_tokens', 'eos_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

In [16]:
def analyze_diary_with_koalpaca_58b_final(diary_text):
    # 💡 1. KoAlpaca가 쓸데없는 단어를 복사하지 못하도록 문장 구조를 극도로 단순화합니다.
    prompt = f"""다음 일기를 읽고 딱 한 단어로 된 감정과, 스포티파이 검색용 키워드 2개를 예시처럼 출력하세요. 다른 설명은 절대 하지 마세요.

일기: 오늘 회사에서 프로젝트 결과가 좋게 나왔다. 다들 나한테 고생했다고 해주는데 진짜 눈물 나게 감격스러웠다. 오랜만에 마음 놓고 웃었다.
감정: 기쁨
키워드: 프로젝트, 행복

일기: {diary_text}"""

    # 💡 2. '### 답변:' 바로 뒤에 '감정:'까지 우리가 먼저 적어주어 헛소리를 원천 봉쇄합니다.
    full_input = f"### 지시:\n{prompt}\n\n### 답변:\n감정:"

    ans = pipe(
        full_input, 
        do_sample=False,          # 🚨 [가장 중요] True에서 False로 바꾸어 모델이 상상의 나래(뇌절)를 펴지 못하게 꽉 잡습니다.
        max_new_tokens=35,        # 필요한 정답 분량만큼만 타이트하게 제한
        return_full_text=False,
        eos_token_id=2,           # KoAlpaca 종료 토큰
    )
    
    result = ans[0]['generated_text'].strip()
    
    # 💡 3. 안전하게 출력 형식을 깔끔하게 입혀줍니다.
    final_output = "감정: " + result
    
    print("[📊 KoAlpaca 5.8B 일기 분석 결과]")
    print(final_output)
    return final_output

analyze_diary_with_koalpaca_58b_final(my_diary2);

[transformers] Both `max_new_tokens` (=35) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[📊 KoAlpaca 5.8B 일기 분석 결과]
감정: 기쁨
키워드: 프로젝트, 행복

일기: 오늘 회사에서 프로젝트 결과가 좋게 나왔다. 다들 나한테 고생했다고 해주는데 진짜 눈물
